# 01 - Ingestão do Brent

## Objetivo
Realizar a ingestão do arquivo bruto da série diária do preço do petróleo Brent, disponibilizado pela U.S. Energy Information Administration (EIA), para a camada Bronze do Lakehouse.

## Origem
Arquivo Excel armazenado no Volume RAW do Databricks.

## Destino
Tabela `workspace.bronze.brent_raw`.

## Leitura do arquivo de origem

A série histórica do Brent foi disponibilizada pela EIA em formato Excel e armazenada no Volume RAW do projeto.

Como o arquivo possui linhas de cabeçalho e metadados antes do início da série histórica, a estrutura original será inspecionada antes da preparação dos dados para persistência na camada Bronze.

In [0]:
caminho_brent = "/Volumes/workspace/raw/dados_raw/brent/brent_daily_eia.xls"

print(caminho_brent)

/Volumes/workspace/raw/dados_raw/brent/brent_daily_eia.xls


### Leitura inicial com Pandas

O arquivo da EIA está no formato Excel legado (`.xls`). A leitura inicial é realizada com Pandas e `xlrd`, o que permite inspecionar a estrutura do arquivo antes da conversão para um DataFrame Spark.

Após essa etapa, os dados seguem o processamento no Spark e são persistidos em formato Delta na camada Bronze.

In [0]:
import pandas as pd

In [0]:
%pip install xlrd

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
df_brent_pd = pd.read_excel(caminho_brent)

In [0]:
df_brent_pd.head(15)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,Workbook Contents,NaN,NaN,NaN,NaN
1,NaN,Europe Brent Spot Price FOB (Dollars per Barrel),NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Click worksheet name or tab at bottom for data,NaN,NaN,NaN,NaN
4,NaN,Worksheet Name,Description,# Of Series,Frequency,Latest Data for
5,NaN,Data 1,Europe Brent Spot Price FOB (Dollars per Barrel),1,Daily,9/9/2026
6,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,Release Date:,9/10/2026,NaN,NaN,NaN
8,NaN,Next Release Date:,9/16/2026,NaN,NaN,NaN
9,NaN,Excel File Name:,rbrted.xls,NaN,NaN,NaN


In [0]:
xls = pd.ExcelFile(caminho_brent)

print(xls.sheet_names)

['Contents', 'Data 1']


In [0]:
df_brent_pd = pd.read_excel(caminho_brent, sheet_name='Data 1',
                            skiprows=2)
df_brent_pd.head(10)

,Date,Europe Brent Spot Price FOB (Dollars per Barrel)
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63
5,1987-05-27,18.60
6,1987-05-28,18.60
7,1987-05-29,18.58
8,1987-06-01,18.65
9,1987-06-02,18.68


In [0]:
print("Dimensões da base:")
print(f"Linhas: {df_brent_pd.shape[0]}")
print(f"Colunas: {df_brent_pd.shape[1]}")

print("\nTipos:")
print(df_brent_pd.dtypes)

print("\nPeríodo:")
print(f"Menor data: {df_brent_pd['Date'].min()}")
print(f"Maior data: {df_brent_pd['Date'].max()}")

print("\nValores nulos:")
print(df_brent_pd.isnull().sum())

Dimensões da base:
Linhas: 9973
Colunas: 2

Tipos:
Date                                                datetime64[ns]
Europe Brent Spot Price FOB (Dollars per Barrel)           float64
dtype: object

Período:
Menor data: 1987-05-20 00:00:00
Maior data: 2026-09-09 00:00:00

Valores nulos:
Date                                                0
Europe Brent Spot Price FOB (Dollars per Barrel)    0
dtype: int64


### Preparação para a camada Bronze

Após a identificação da estrutura útil do arquivo, os dados são convertidos para um DataFrame Spark.

Nesta etapa são mantidas apenas as informações que representam a série histórica do Brent — data e preço — sem aplicação de regras de negócio ou transformações analíticas. O arquivo original permanece preservado no Volume RAW.

In [0]:
df_brent_spark = spark.createDataFrame(df_brent_pd)

df_brent_spark.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Europe Brent Spot Price FOB (Dollars per Barrel): double (nullable = true)



In [0]:
df_brent_spark = (
    df_brent_spark
    .withColumnRenamed("Europe Brent Spot Price FOB (Dollars per Barrel)",
                       "Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel")
)
    
df_brent_spark.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Europe_Brent_Spot_Price_FOB_Dollars_per_Barrel: double (nullable = true)



## Persistência na camada Bronze

O DataFrame é persistido em formato Delta na tabela `workspace.bronze.brent_raw`.

Após a gravação, a tabela é novamente carregada e sua quantidade de registros é comparada à do DataFrame de origem, permitindo verificar se todos os registros preparados para a ingestão foram persistidos.

In [0]:
(
    df_brent_spark.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.brent_raw")
)

print("Tabela criada com sucesso")

Tabela criada com sucesso


In [0]:
df_brent_bronze = spark.table("workspace.bronze.brent_raw")

print(f"Registros no DataFrame de origem: {df_brent_spark.count()}")
print(f"Registros na tabela: {df_brent_bronze.count()}")

Registros no DataFrame de origem: 9973
Registros na tabela: 9973
